# Reference Extraction Audit — Thống kê & tìm nguyên nhân không trích xuất được

**Mục tiêu:** Chạy trực tiếp `reference_extractor.py` của nhóm trên `merged.tex` trong từng version, sau đó thử fallback `.bib/.bbl` **đúng như `main.py` hiện tại**. Tạo bảng: **(1)** bài/phiên bản nào lấy được reference, **(2)** bao nhiêu reference, **(3)** bài/phiên bản nào không lấy được và **lý do nghi vấn**, **(4)** các trường hợp cần xem lại regex.

**Quan trọng:**
- `extracted_count > 0` có nghĩa **code trả về entry**, không phải chứng nhận entry đúng hoặc đủ.
- `extracted_count == 0` có nghĩa **code chưa lấy được reference**, **không chứng minh** bài không có references.
- Lý do `REGEX_*`, `*_SUSPECTED` là **chẩn đoán heuristic**, không phải nguyên nhân đã xác minh.
- Notebook **chỉ đo extraction**, không chạy dedup/export, không sửa file nguồn hay ghi đè `references.bib`.
- Đơn vị thống kê là **publication** (paper) và **version** (v1, v2...). Các version không được tạo thành thư mục trong đầu vào **không thể tự suy ra** và cần kiểm tra riêng với metadata từ Milestone 1.

### Chuẩn bị
1. Đặt notebook trong `reference_pipeline/` hoặc chạy từ thư mục dự án, đảm bảo tìm thấy `reference_extractor.py`, `main.py`.
2. Chỉnh `ROOT_DIR` / `BBL_DIR` trong ô cấu hình. Mặc định dùng đường dẫn giống `main.py`.
3. Cần `pandas` và `matplotlib` để xem bảng/biểu đồ; có thể cài bằng `pip install pandas matplotlib tqdm` nếu thiếu. Pipeline trích xuất của nhóm vẫn chỉ dùng Python standard library.
4. Chạy lần lượt **Run All**; bảng và CSV sẽ nằm trong `REPORT_DIR`.

In [6]:
from pathlib import Path
import sys, re, json, time
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = lambda xs, **kwargs: xs

# ===== CẤU HÌNH: CHỈNH Ở ĐÂY =====
ROOT_DIR = Path('Project_IntroData/parsed_structure/parsed_structure')
BBL_DIR = Path('Project_IntroData/bbl')  # Đặt None nếu không có kho bbl fallback
REPORT_DIR = Path('reference_audit_output')
MAX_PAPERS = None  # None = toàn bộ; thử 100 trước khi chạy 20.000
SHOW_N = 25             # Số dòng xem trước
# =================================

def find_pipeline_dir():
    """Dò thư mục chứa các module thật của nhóm, không chép lại extractor."""
    for base in (Path.cwd(), *Path.cwd().parents):
        for suffix in ('', 'reference_pipeline', 'src/reference_pipeline'):
            folder = (base / suffix).resolve()
            if (folder / 'reference_extractor.py').is_file() and (folder / 'main.py').is_file():
                return folder
    raise FileNotFoundError(
        'Không tìm thấy reference_extractor.py và main.py. '
        'Hãy đặt notebook trong reference_pipeline/ hoặc mở Jupyter từ thư mục dự án.'
    )

PIPELINE_DIR = find_pipeline_dir()
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))

import reference_extractor as rex
import main as pipeline_main

# Phòng trường hợp Jupyter được mở từ root, src hoặc reference_pipeline.
def resolve_existing(path):
    if path is None:
        return None
    path = Path(path).expanduser()
    if path.is_absolute():
        return path
    for base in (Path.cwd(), PIPELINE_DIR.parent, PIPELINE_DIR.parent.parent):
        candidate = (base / path).resolve()
        if candidate.exists():
            return candidate
    return (Path.cwd() / path).resolve()

ROOT_DIR = resolve_existing(ROOT_DIR)
BBL_DIR = resolve_existing(BBL_DIR)
REPORT_DIR = (Path.cwd() / REPORT_DIR).resolve()
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print('Pipeline:', PIPELINE_DIR)
print('Dữ liệu:', ROOT_DIR)
print('Kho .bib/.bbl:', BBL_DIR if (BBL_DIR and BBL_DIR.exists()) else 'Không tìm thấy / tắt')
print('Báo cáo:', REPORT_DIR)
if not ROOT_DIR.is_dir():
    raise FileNotFoundError(
        f'ROOT_DIR chưa tồn tại: {ROOT_DIR}\n'
        'Hãy chỉnh ROOT_DIR thành thư mục gồm các paper/<version>/merged.tex.'
    )

Pipeline: D:\University\Year 3\NMKHDL\milestone1\Project_IntroData\reference_pipeline
Dữ liệu: D:\University\Year 3\NMKHDL\milestone1\Project_IntroData\parsed_structure\parsed_structure
Kho .bib/.bbl: D:\University\Year 3\NMKHDL\milestone1\Project_IntroData\bbl
Báo cáo: D:\University\Year 3\NMKHDL\milestone1\Project_IntroData\reference_pipeline\reference_audit_output


## 1. Các dấu hiệu để chẩn đoán lỗi

Các regex trong phần audit dưới đây **chỉ phục vụ việc chẩn đoán**, **không dùng để thay thế** các regex hiện hành trong `reference_extractor.py`. Số lượng trích xuất lấy từ chính hàm `rex.extract_bibitems()` và `pipeline_main.extract_from_bbl_dir()`.

Cần phân biệt **không khớp regex** với **thiếu dữ liệu** hoặc **có heading nhưng không tách được entries**. Phân loại lý do có tính ưu tiên: lỗi đọc > file đầu vào thiếu > dấu hiệu references nhưng không lấy được > không thấy dấu hiệu nào.

In [7]:
# Dấu hiệu bổ sung để giúp giải thích các ca extractor trả về []
CITATION_SIGNAL_RE = re.compile(
    r'\\(?:cite[a-zA-Z]*|nocite|parencite|textcite|autocite)\b\s*(?:\[[^]]*\]\s*){0,2}\{', re.I)
BIB_COMMAND_RE = re.compile(r'\\(?:bibliography|addbibresource|printbibliography)\b', re.I)
BIB_ENV_SIGNAL_RE = re.compile(r'\\begin\s*\{(?:thebibliography|references|biblist)\}', re.I)
PLAIN_HEADING_RE = re.compile(
    r'(?im)^\s*(?:\\(?:section|chapter|subsection)\*?\s*\{\s*)?'
    r'(?:\\(?:textbf|textsc)\s*\{\s*)?'
    r'(?:references?|bibliography|works cited|literature cited)\s*\}*\s*$'
)
NUMBERED_LINE_RE = re.compile(r'(?m)^\s*(?:\[\s*\d+\s*\]|\(\s*\d+\s*\)|\d+\.)\s+\S')
YEAR_RE = re.compile(r'\b(?:19|20)\d{2}\b')

def tex_signals(cleaned_text):
    """Tín hiệu hỗ trợ giải thích (không thay extractor)."""
    heading = rex.REF_HEADING_RE.search(cleaned_text)
    possible_heading = PLAIN_HEADING_RE.search(cleaned_text)
    near_heading = cleaned_text[heading.end():heading.end()+900] if heading else ''
    return {
        'has_ref_heading_regex': bool(heading),
        'has_plain_heading': bool(possible_heading),
        'has_cite_command': bool(CITATION_SIGNAL_RE.search(cleaned_text)),
        'has_bibliography_command': bool(BIB_COMMAND_RE.search(cleaned_text)),
        'has_bibliography_environment': bool(BIB_ENV_SIGNAL_RE.search(cleaned_text)),
        'has_numbered_reference_candidates': bool(NUMBERED_LINE_RE.search(near_heading)),
        'years_near_ref_heading': len(YEAR_RE.findall(near_heading)),
        'ref_section_preview': re.sub(r'\s+', ' ', near_heading)[:280],
    }

def identify_tex_method(cleaned_text, tex_path):
    """Xác định nhánh ĐẦU TIÊN trả về refs giống thứ tự hàm extract_bibitems.
    Chỉ dùng khi đã biết extract_bibitems trả về >0; không thay đổi logic gốc.
    """
    if rex.EXPLICIT_KEY_RE.search(cleaned_text) or rex.HARVARD_RE.search(cleaned_text):
        return 'explicit_macro (bibitem/reference/refitem/harvarditem)'
    if rex.AMSREFS_RE.search(cleaned_text):
        return 'amsrefs'
    if rex.BIBTEX_ENTRY_RE.search(cleaned_text):
        return 'embedded_bibtex'
    if rex._extract_list_environments(cleaned_text, str(tex_path)):
        return 'list_environment'
    if rex._extract_aas_references(cleaned_text, str(tex_path)):
        return 'aas_reference'
    if rex._extract_section_references(cleaned_text, str(tex_path)):
        return 'free_form_ref_heading'
    return 'UNDETERMINED (needs inspection)'

def fallback_info(paper_id, version):
    """Bám đúng chiến lược chọn folder và loại file trong main.extract_from_bbl_dir."""
    if BBL_DIR is None:
        return 'disabled', []
    paper_bbl = BBL_DIR / paper_id
    if not paper_bbl.is_dir():
        return 'missing', []
    chosen = paper_bbl / version
    if not chosen.exists():
        chosen = paper_bbl
    bibs = sorted(chosen.glob('*.bib'))
    if bibs:
        return 'bib', bibs
    bbls = sorted(chosen.glob('*.bbl'))
    if bbls:
        return 'bbl', bbls
    return 'no_files_in_selected_folder', []

def reason_for_zero(row):
    """Trả (reason_code, mô tả). KHÔNG khẳng định đây là ground truth."""
    if row['error']:
        return ('EXTRACTION_ERROR', 'Lỗi đọc hoặc xử lý nguồn: ' + row['error'][:160])
    if not row['merged_tex_exists']:
        return ('MISSING_MERGED_TEX', 'Không có merged.tex và fallback không trả reference')
    if row['tex_size_bytes'] == 0:
        return ('EMPTY_MERGED_TEX', 'merged.tex rỗng và fallback không trả reference')
    if row['fallback_file_count'] and row['fallback_count'] == 0:
        return ('FALLBACK_PARSE_EMPTY', 'Có .bib/.bbl fallback nhưng code không trích xuất được entry')
    if row['local_bib_count'] or row['local_bbl_count']:
        return ('LOCAL_BIB_BBL_NOT_READ', 'Có .bib/.bbl trong version nhưng main.py chỉ fallback ở BBL_DIR')
    if row['has_ref_heading_regex']:
        if row['has_numbered_reference_candidates'] or row['years_near_ref_heading']:
            return ('REF_HEADING_SPLIT_FAILED', 'Có heading References khớp regex, có dấu hiệu entries nhưng các cách tách đều trả []')
        return ('REF_HEADING_NO_PARSEABLE_ITEMS', 'Có heading References nhưng chưa nhận diện được entry (có thể rỗng hoặc cách viết khác)')
    if row['has_plain_heading']:
        return ('HEADING_NOT_COVERED_BY_REGEX', 'Có heading giống References/Works Cited nhưng REF_HEADING_RE không khớp')
    if row['has_bibliography_environment']:
        return ('BIB_ENV_NOT_EXTRACTED', 'Có môi trường bibliography/references nhưng extractor không lấy được entry')
    if row['has_bibliography_command']:
        return ('BIB_SOURCE_UNAVAILABLE', 'Có lệnh bibliography/addbibresource/printbibliography nhưng code không lấy được .bib/.bbl')
    if row['has_cite_command']:
        return ('HAS_CITES_BUT_NO_REFS', 'Có lệnh cite nhưng không trích xuất được danh mục references')
    return ('NO_REFERENCE_SIGNALS', 'Không phát hiện dấu hiệu references bằng các kiểm tra hiện tại; chưa thể kết luận không có references')

## 2. Chạy audit toàn bộ dataset

- Cách quét giống `main.py`: `ROOT_DIR/<paper>/<version>/merged.tex`; chỉ dùng `BBL_DIR/<paper>/<version>` (hoặc `BBL_DIR/<paper>`) làm fallback khi tex trả `[]`.
- Mỗi version được xử lý độc lập. Không chạy dedup, nên tổng refs giữa các phiên bản **có thể trùng**.
- Nếu thiếu file `merged.tex` nhưng `.bib`/`.bbl` fallback dùng được, version vẫn tính là trích xuất được; dấu hiệu thiếu source được lưu để báo cáo.

In [ ]:
def scan_version(paper_dir, version_dir):
    paper_id, version = paper_dir.name, version_dir.name
    tex = version_dir / 'merged.tex'
    tex_exists = tex.is_file()
    row = {
        'arxiv_id': paper_id,
        'version': version,
        'merged_tex_path': str(tex),
        'merged_tex_exists': tex_exists,
        'tex_size_bytes': tex.stat().st_size if tex_exists else None,
        'tex_count': 0,
        'fallback_count': 0,
        'extracted_count': 0,
        'source': 'none',
        'extractor_method': '',
        'reason_code': '',
        'reason_detail': '',
        'error': '',
        'fallback_kind': '',
        'fallback_file_count': 0,
        'fallback_paths': '',
        'local_bib_count': len(list(version_dir.glob('*.bib'))),
        'local_bbl_count': len(list(version_dir.glob('*.bbl'))),
        'empty_raw_entries': 0,
        'suspicious_mixed_sources': False,
        'has_ref_heading_regex': False,
        'has_plain_heading': False,
        'has_cite_command': False,
        'has_bibliography_command': False,
        'has_bibliography_environment': False,
        'has_numbered_reference_candidates': False,
        'years_near_ref_heading': 0,
        'ref_section_preview': '',
    }
    try:
        raw = []
        if tex_exists:
            # CHẠY HÀM THẬT, KHÔNG TỰ VIẾT LẠI EXTRACTOR
            raw = rex.extract_bibitems(tex)
            row['tex_count'] = len(raw)
            # Diagnostic làm trên cùng nội dung sau khi bỏ comment như extractor.
            text = rex.remove_latex_comments(tex.read_text(encoding='utf-8', errors='replace'))
            row.update(tex_signals(text))
            if raw:
                row['source'] = 'tex'
                row['extractor_method'] = identify_tex_method(text, tex)
                # Trường hợp nguồn bị trộn; main dừng ở nhánh trả refs đầu tiên.
                row['suspicious_mixed_sources'] = bool(
                    row['has_bibliography_command'] and row['extractor_method'] != 'embedded_bibtex'
                )
        # Chỉ fallback khi merged.tex KHÔNG lấy được ref, giống main.py.
        if not raw and BBL_DIR is not None:
            kind, sources = fallback_info(paper_id, version)
            row['fallback_kind'] = kind
            row['fallback_file_count'] = len(sources)
            row['fallback_paths'] = '; '.join(str(p) for p in sources)
            # Hàm của main.py trả về các dict (từ .bib hoặc đã parse .bbl).
            raw = pipeline_main.extract_from_bbl_dir(BBL_DIR / paper_id, version)
            row['fallback_count'] = len(raw)
            if raw:
                row['source'] = 'fallback_' + kind
                row['extractor_method'] = 'parse_bib_file' if kind == 'bib' else 'extract_bibitems(.bbl)'
        row['extracted_count'] = len(raw)
        row['empty_raw_entries'] = sum(not str(ref.get('raw_content') or '').strip() for ref in raw)
    except Exception as exc:
        # Tiếp tục quét các version còn lại, không crash toàn bộ 20.000 paper.
        row['error'] = f'{type(exc).__name__}: {exc}'
        row['source'] = 'error'
        row['extracted_count'] = 0  # Không tính dữ liệu dở dang là thành công.
    if row['extracted_count'] == 0:
        row['reason_code'], row['reason_detail'] = reason_for_zero(row)
    else:
        row['reason_code'] = 'EXTRACTED'
        row['reason_detail'] = ('Có entry trích xuất, chưa kiểm chứng precision/recall hoặc metadata')
    return row

def scan_dataset(root, max_papers=None):
    paper_dirs = sorted(p for p in root.iterdir() if p.is_dir())
    if max_papers is not None:
        paper_dirs = paper_dirs[:max_papers]
    rows = []
    empty_paper_dirs = []
    t0 = time.perf_counter()
    for paper_dir in tqdm(paper_dirs, desc='Đang kiểm tra publications'):
        versions = sorted(p for p in paper_dir.iterdir() if p.is_dir())
        if not versions:
            empty_paper_dirs.append(paper_dir.name)
        for version_dir in versions:
            rows.append(scan_version(paper_dir, version_dir))
    elapsed = time.perf_counter() - t0
    return pd.DataFrame(rows), paper_dirs, empty_paper_dirs, elapsed

version_df, scanned_papers, empty_paper_dirs, elapsed_seconds = scan_dataset(ROOT_DIR, MAX_PAPERS)
if version_df.empty:
    print('Không thấy thư mục version để quét. Kiểm tra layout ROOT_DIR/<arxiv_id>/<version>/merged.tex')
else:
    print(f'Đã duyệt {len(scanned_papers):,} paper dirs, {len(version_df):,} version dirs trong {elapsed_seconds:.2f}s')
    print(f'Thư mục paper không có version: {len(empty_paper_dirs):,}')
    display(version_df[['arxiv_id','version','extracted_count','source','extractor_method','reason_code']].head(SHOW_N))

Đang kiểm tra publications:  14%|█▎        | 2736/20000 [05:38<17:33, 16.39it/s]  

## 3. Thống kê thành công/thất bại theo version và theo bài báo

- **Paper có refs:** có ít nhất một version mà extractor/fallback trả > 0 entry.
- **Paper không trích xuất được:** tất cả version đã quét đều trả 0. Đây **không phải** khẳng định bài báo không có bibliography.
- **Paper trích xuất một phần:** có version trả > 0 và version khác trả 0.
- `No version directory`: báo riêng vì không đủ thông tin để tính coverage từng version.

In [4]:
if not version_df.empty:
    version_df['has_extracted_refs'] = version_df['extracted_count'] > 0
    by_paper = version_df.groupby('arxiv_id', sort=True).agg(
        n_versions=('version','size'),
        n_versions_with_refs=('has_extracted_refs','sum'),
        n_versions_without_refs=('has_extracted_refs', lambda s: (~s).sum()),
        n_raw_refs_across_versions=('extracted_count','sum'),
        n_errors=('error', lambda s: (s != '').sum()),
    ).reset_index()
    def paper_status(r):
        if r['n_versions_with_refs'] == 0: return 'NO_VERSION_EXTRACTED'
        if r['n_versions_without_refs'] == 0: return 'ALL_VERSIONS_EXTRACTED'
        return 'PARTIAL_VERSIONS_EXTRACTED'
    by_paper['status'] = by_paper.apply(paper_status, axis=1)
    paper_df = by_paper
    summary = {
        'paper_directories_scanned': int(len(scanned_papers)),
        'papers_with_version_directories': int(len(paper_df)),
        'paper_directories_without_versions': int(len(empty_paper_dirs)),
        'version_directories_scanned': int(len(version_df)),
        'versions_extracted': int(version_df['has_extracted_refs'].sum()),
        'versions_not_extracted': int((~version_df['has_extracted_refs']).sum()),
        'versions_extracted_pct': round(100 * version_df['has_extracted_refs'].mean(), 2),
        'papers_with_refs': int((paper_df['n_versions_with_refs'] > 0).sum()),
        'papers_without_extracted_refs': int((paper_df['n_versions_with_refs'] == 0).sum()),
        'papers_all_versions_extracted': int((paper_df['status'] == 'ALL_VERSIONS_EXTRACTED').sum()),
        'papers_partially_extracted': int((paper_df['status'] == 'PARTIAL_VERSIONS_EXTRACTED').sum()),
        'total_extracted_entries_before_dedup': int(version_df['extracted_count'].sum()),
        'versions_with_extraction_errors': int((version_df['error'] != '').sum()),
        'elapsed_seconds': round(elapsed_seconds, 2),
    }
    display(pd.DataFrame([summary]).T.rename(columns={0:'Giá trị'}))
    display(paper_df.head(SHOW_N))
else:
    paper_df = pd.DataFrame(columns=['arxiv_id','n_versions','n_versions_with_refs',
                                     'n_versions_without_refs','n_raw_refs_across_versions','n_errors','status'])
    summary = {'paper_directories_scanned': len(scanned_papers), 'version_directories_scanned': 0}

,Giá trị
paper_directories_scanned,100.00
papers_with_version_directories,100.00
paper_directories_without_versions,0.00
version_directories_scanned,152.00
versions_extracted,90.00
versions_not_extracted,62.00
versions_extracted_pct,59.21
papers_with_refs,59.00
papers_without_extracted_refs,41.00
papers_all_versions_extracted,57.00


,arxiv_id,n_versions,n_versions_with_refs,n_versions_without_refs,n_raw_refs_across_versions,n_errors,status
0,2511-11709,1,1,0,105,0,ALL_VERSIONS_EXTRACTED
1,2511-11710,1,1,0,54,0,ALL_VERSIONS_EXTRACTED
2,2511-11711,1,1,0,17,0,ALL_VERSIONS_EXTRACTED
3,2511-11712,1,1,0,14,0,ALL_VERSIONS_EXTRACTED
4,2511-11713,1,1,0,161,0,ALL_VERSIONS_EXTRACTED
5,2511-11714,1,1,0,36,0,ALL_VERSIONS_EXTRACTED
6,2511-11715,1,1,0,103,0,ALL_VERSIONS_EXTRACTED
7,2511-11716,1,1,0,66,0,ALL_VERSIONS_EXTRACTED
8,2511-11717,1,1,0,127,0,ALL_VERSIONS_EXTRACTED
9,2511-11718,1,1,0,20,0,ALL_VERSIONS_EXTRACTED


## 4. Bài nào không lấy được refs, và nghi ngờ vì sao?

Đọc bảng `no_refs` theo `reason_code` / `reason_detail`. Các ca có `HEADING_NOT_COVERED_BY_REGEX` hoặc `REF_HEADING_SPLIT_FAILED` nên được dùng để cải tiến `reference_extractor.py` trước. Các ca `NO_REFERENCE_SIGNALS` cần lấy mẫu để phân biệt bài thực sự không có tài liệu tham khảo với các định dạng chưa được phát hiện.

In [5]:
if not version_df.empty:
    no_refs = version_df.loc[~version_df['has_extracted_refs']].copy()
    reason_counts = (no_refs.groupby(['reason_code', 'reason_detail'], dropna=False)
                     .size().reset_index(name='version_count')
                     .sort_values('version_count', ascending=False))
    display(reason_counts)
    display(no_refs[['arxiv_id','version','reason_code','reason_detail',
                     'has_ref_heading_regex','has_plain_heading',
                     'has_cite_command','has_bibliography_command',
                     'fallback_kind','local_bib_count','ref_section_preview']].head(SHOW_N))
else:
    no_refs = pd.DataFrame()
    reason_counts = pd.DataFrame(columns=['reason_code','reason_detail','version_count'])

,reason_code,reason_detail,version_count
0,BIB_SOURCE_UNAVAILABLE,Có lệnh bibliography/addbibresource/printbibli...,32
1,EMPTY_MERGED_TEX,merged.tex rỗng và fallback không trả reference,19
3,REF_HEADING_NO_PARSEABLE_ITEMS,Có heading References nhưng chưa nhận diện đượ...,9
2,HAS_CITES_BUT_NO_REFS,Có lệnh cite nhưng không trích xuất được danh ...,2


,arxiv_id,version,reason_code,reason_detail,has_ref_heading_regex,has_plain_heading,has_cite_command,has_bibliography_command,fallback_kind,local_bib_count,ref_section_preview
12,2511-11721,v1,EMPTY_MERGED_TEX,merged.tex rỗng và fallback không trả reference,False,False,False,False,no_files_in_selected_folder,0,
13,2511-11721,v2,EMPTY_MERGED_TEX,merged.tex rỗng và fallback không trả reference,False,False,False,False,no_files_in_selected_folder,0,
14,2511-11722,v1,EMPTY_MERGED_TEX,merged.tex rỗng và fallback không trả reference,False,False,False,False,no_files_in_selected_folder,0,
15,2511-11722,v2,EMPTY_MERGED_TEX,merged.tex rỗng và fallback không trả reference,False,False,False,False,no_files_in_selected_folder,0,
16,2511-11723,v1,EMPTY_MERGED_TEX,merged.tex rỗng và fallback không trả reference,False,False,False,False,no_files_in_selected_folder,0,
17,2511-11723,v2,EMPTY_MERGED_TEX,merged.tex rỗng và fallback không trả reference,False,False,False,False,no_files_in_selected_folder,0,
34,2511-11738,v1,EMPTY_MERGED_TEX,merged.tex rỗng và fallback không trả reference,False,False,False,False,no_files_in_selected_folder,0,
35,2511-11738,v2,EMPTY_MERGED_TEX,merged.tex rỗng và fallback không trả reference,False,False,False,False,no_files_in_selected_folder,0,
38,2511-11741,v1,EMPTY_MERGED_TEX,merged.tex rỗng và fallback không trả reference,False,False,False,False,no_files_in_selected_folder,0,
54,2511-11748,v1,EMPTY_MERGED_TEX,merged.tex rỗng và fallback không trả reference,False,False,False,False,no_files_in_selected_folder,0,


## 5. Những regex/nhánh nào đang trích xuất được nhiều nhất?

`extractor_method` là **nhánh đầu tiên** trả refs. Vì `extract_bibitems()` trả ngay khi tìm được kết quả, **không thể từ biểu đồ này kết luận các nhánh phía sau không có reference**.

In [ ]:
if not version_df.empty:
    methods = (version_df.loc[version_df['has_extracted_refs']]
               .groupby(['source','extractor_method'], dropna=False)
               .agg(versions=('version','size'), entries=('extracted_count','sum'))
               .reset_index().sort_values('versions', ascending=False))
    display(methods)

    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
    version_df['has_extracted_refs'].value_counts().rename(
        index={True: 'Extracted > 0', False: 'Extracted = 0'}
    ).plot(kind='bar', ax=axes[0], title='Số version theo trạng thái', rot=0)
    if not reason_counts.empty:
        reason_counts.set_index('reason_code')['version_count'].sort_values().plot(
            kind='barh', ax=axes[1], title='Nguyên nhân nghi vấn (version chưa lấy refs)')
    else:
        axes[1].text(.2,.5,'Không có version chưa lấy refs', transform=axes[1].transAxes)
    axes[0].set_ylabel('Số version')
    axes[1].set_xlabel('Số version')
    plt.tight_layout()
    plt.show()
else:
    methods = pd.DataFrame(columns=['source','extractor_method','versions','entries'])

## 6. Hàng đợi review ưu tiên (chỉ kiểm tra ca nghi vấn)

Không chỉ những version có `extracted_count == 0` mới đáng chú ý. Một số version **có > 0 refs nhưng** có entry rỗng hoặc có nhiều nguồn bibliography bị pipeline bỏ qua do ưu tiên dừng ở nhánh đầu tiên. Danh sách dưới đây giúp chọn đúng ca để mở LaTeX kiểm tra, thay vì duyệt tất cả 20.000 bài.

In [ ]:
if not version_df.empty:
    HIGH_PRIORITY = {
        'EXTRACTION_ERROR', 'REF_HEADING_SPLIT_FAILED',
        'HEADING_NOT_COVERED_BY_REGEX', 'BIB_ENV_NOT_EXTRACTED',
        'FALLBACK_PARSE_EMPTY', 'LOCAL_BIB_BBL_NOT_READ',
        'BIB_SOURCE_UNAVAILABLE', 'HAS_CITES_BUT_NO_REFS',
    }
    def priority(r):
        if r['error']: return 'P0_ERROR'
        if r['extracted_count'] == 0 and r['reason_code'] in HIGH_PRIORITY: return 'P1_LIKELY_MISSED'
        if r['extracted_count'] > 0 and r['empty_raw_entries'] > 0: return 'P2_EMPTY_EXTRACTED_ENTRY'
        if r['extracted_count'] > 0 and r['suspicious_mixed_sources']: return 'P2_MIXED_SOURCE'
        if r['extracted_count'] == 0: return 'P3_NEEDS_SAMPLING'
        return ''
    version_df['review_priority'] = version_df.apply(priority, axis=1)
    review_queue = version_df.loc[version_df['review_priority'] != ''].copy()
    order = ['P0_ERROR','P1_LIKELY_MISSED','P2_EMPTY_EXTRACTED_ENTRY',
             'P2_MIXED_SOURCE','P3_NEEDS_SAMPLING']
    review_queue['priority_rank'] = review_queue['review_priority'].map({v:i for i,v in enumerate(order)})
    review_queue = review_queue.sort_values(['priority_rank','reason_code','arxiv_id','version'])
    display(review_queue[['review_priority','arxiv_id','version','extracted_count',
                          'reason_code','reason_detail','extractor_method',
                          'merged_tex_path']].head(SHOW_N))
else:
    review_queue = pd.DataFrame()

## 7. Tra cứu nhanh một bài cụ thể

Sau khi có `no_refs`, đặt `INSPECT_ARXIV_ID` và `INSPECT_VERSION` để nhìn toàn bộ tín hiệu và preview của version. **Không tự động thay code hoặc sửa source.**

In [ ]:
INSPECT_ARXIV_ID = ''     # VD: '2511-11709'
INSPECT_VERSION = ''      # VD: 'v1'; để trống để lấy mọi version của paper

if not version_df.empty and INSPECT_ARXIV_ID.strip():
    selected = version_df[version_df['arxiv_id'] == INSPECT_ARXIV_ID.strip()]
    if INSPECT_VERSION.strip():
        selected = selected[selected['version'] == INSPECT_VERSION.strip()]
    if selected.empty:
        print('Không có paper/version này trong dữ liệu vừa quét.')
    else:
        display(selected.T)
        for _, r in selected.iterrows():
            print('\n', r['arxiv_id'], r['version'])
            print('Tín hiệu gần heading:', r['ref_section_preview'] or '(Không tìm thấy heading bằng REF_HEADING_RE)')
else:
    print('Nhập INSPECT_ARXIV_ID để tra cứu, sau đó chạy lại cell này.')

## 8. Xuất CSV / JSON để nhóm phân chia kiểm tra và báo cáo Jupyter Book

Notebook không sửa `main.py`, `reference_extractor.py` hoặc các source LaTeX. Toàn bộ báo cáo nằm trong `REPORT_DIR`. Không commit corpus hoặc CSV lớn vào repository nếu môn học chỉ cho nộp source code.

In [ ]:
output_files = {}
def save_csv(name, data):
    destination = REPORT_DIR / name
    data.to_csv(destination, index=False, encoding='utf-8-sig')
    output_files[name] = str(destination)

save_csv('reference_versions_audit.csv', version_df)
save_csv('reference_papers_summary.csv', paper_df)
save_csv('reference_no_refs_versions.csv', no_refs)
save_csv('reference_reason_counts.csv', reason_counts)
save_csv('reference_extraction_methods.csv', methods)
save_csv('reference_review_queue.csv', review_queue)

if empty_paper_dirs:
    save_csv('paper_dirs_without_versions.csv', pd.DataFrame({'arxiv_id':empty_paper_dirs}))
summary['unscannable_paper_directories'] = empty_paper_dirs[:100]  # Tối đa 100 ví dụ trong JSON
summary['unscannable_paper_directory_count'] = len(empty_paper_dirs)
summary_path = REPORT_DIR / 'reference_audit_summary.json'
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
output_files[summary_path.name] = str(summary_path)

print('Đã tạo các file báo cáo:')
for name, path in output_files.items():
    print(f' - {name}: {path}')

## 9. Đọc số liệu cho đúng khi viết báo cáo

1. **Coverage của extractor** = số version có ít nhất 1 entry trích xuất / tổng số version đã quét. Không gọi chỉ số này là *reference recall* nếu chưa gán nhãn ground truth.
2. **Publication coverage** = số bài có ít nhất một version trích xuất được / số bài có thư mục version.
3. **Phân loại lỗi** trong `reason_code` là **nguyên nhân nghi vấn**, cần xác nhận bằng một tập mẫu có nhãn.
4. `NO_REFERENCE_SIGNALS` không chắc là bài không có References. Hãy lấy mẫu để phát hiện các mẫu chưa được regex hỗ trợ.
5. Các paper có thư mục version bị thiếu hoàn toàn không được nhìn thấy bởi parser hiện tại; đối chiếu inventory Milestone 1 để phát hiện.
6. Nếu `.bib` nằm ngay trong `version_dir` nhưng không ở cấu trúc `BBL_DIR` theo `main.py`, notebook đánh dấu `LOCAL_BIB_BBL_NOT_READ`. Đây là **giới hạn đường dẫn fallback**, không phải vì regex của LaTeX không nhận diện được.
7. Nếu muốn thống kê **số references duy nhất sau dedup**, cần chạy riêng `reference_deduplicator.py` sau khi extraction hoàn tất; chỉ số `total_extracted_entries_before_dedup` hiện tại đếm cả refs ở nhiều version.